# Diabetes Health Indicators — Data Cleaning & EDA & Baseline model
**Dataset:** diabetes_binary_5050split_health_indicators_BRFSS2015.csv

**Repository note.** This notebook preserves the submitted project analysis while using repository-relative data paths. Raw datasets are not redistributed; see `data/README.md`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, recall_score, f1_score,
    roc_auc_score, classification_report
)
print("Libraries loaded successfully.")

In [ ]:
df = pd.read_csv("data/raw/diabetes_binary_5050split_health_indicators_BRFSS2015.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe().T

---
## 1. Data Cleaning

### 1.1 Variable classification

Detail about the variables in dataset:

**Binary (0/1):** Diabetes_binary, HighBP, HighChol, CholCheck, Smoker, Stroke, HeartDiseaseorAttack, PhysActivity, Fruits, Veggies, HvyAlcoholConsump, AnyHealthcare, NoDocbcCost, DiffWalk, Sex

**Ordinal/Count:** GenHlth, MentHlth, PhysHlth, Age, Education, Income

**Continuous:** BMI

> **Note on Age, Education, Income:** These are *ordinal codes*, not actual numeric values.
> For example, Age=1 means 18–24 years; Age=13 means 80+ years.

### 1.2 Missing values

In [ ]:
missing = df.isnull().sum()
print("Total missing values:", missing.sum())

> **Note:** No missing values found. No imputation required.

### 1.3 Duplicate rows

In [ ]:
n_dupes = df.duplicated().sum()
per_dups = n_dupes/len(df)*100
print(f"Duplicate rows: {n_dupes} ({per_dups:.2f}% of dataset)")

> **Note: Duplicates are retained.**
> This is a population-level survey with no unique patient identifier (no ID, DOB, or name).
> Two respondents sharing identical answers for all 21 variables is statistically plausible
> in a large survey sample. Dropping them would risk removing genuine observations.

### 1.4 BMI outlier investigation

BMI is the only truly continuous variable. The raw distribution shows a maximum of 98,
which warrants closer inspection against the CDC source data documentation.

In [ ]:
print("BMI summary statistics:")
print(df["BMI"].describe().round(2))
print()

q1, q3 = df["BMI"].quantile([0.25, 0.75])
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
p99 = df["BMI"].quantile(0.99)

print(f"IQR upper fence (Q3 + 1.5·IQR): {upper_fence:.1f}")
print(f"99th percentile:                 {p99:.1f}")
print(f"Rows with BMI > {upper_fence:.0f} (IQR outliers): {(df['BMI'] > upper_fence).sum()}")
print(f"Rows with BMI > 60:              {(df['BMI'] > 60).sum()} ({(df['BMI'] > 60).mean()*100:.2f}%)")
print(f"Rows with BMI > 80:              {(df['BMI'] > 80).sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: full distribution
axes[0].hist(df["BMI"], bins=30)
axes[0].set_title("BMI distribution (full range)")
axes[0].set_xlabel("BMI")
axes[0].set_ylabel("Count")
axes[0].legend()

# Right: extreme tail only
tail = df[df["BMI"] > 55]["BMI"]
axes[1].hist(tail, bins=30, color="#FA765A")
axes[1].set_title("BMI > 55 (extreme tail detail)")
axes[1].set_xlabel("BMI")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"\nExtreme tail (BMI > 55): {len(tail)} rows = {len(tail)/len(df)*100:.3f}% of data")

#### CDC codebook context

According to the **BRFSS 2015 Codebook** (`_BMI5`), the variable is stored as an integer
with **2 implied decimal places** — meaning a raw value of `9800` corresponds to BMI = **98.00**.
Valid range is documented as **1 – 9999** (i.e., BMI 0.01 – 99.99). This confirms that
BMI = 98 in the Kaggle-processed dataset is **within the documented valid range** and is not
a data entry error or processing artifact.

Clinically, a BMI ≥ 40 is Class III (severe) obesity; values above 60–70 are extremely rare
but do occur in morbidly obese individuals.

> **Decision: No rows are dropped.**
> Values are within the CDC-documented valid range and represent genuine extreme observations.
> Their small proportion (< 0.4% above BMI 60) means they are unlikely to distort model
> performance significantly. We flag them and note that tree-based models (Random Forest,
> XGBoost) are robust to outliers by design.

### 1.5 Cleaning summary

| Check | Finding | Decision |
|-------|---------|----------|
| Missing values | 0 across all 22 columns | No action required |
| Duplicate rows | 1,635 (2.31%) | Retained — no unique patient ID exists |
| BMI outliers | 260 rows with BMI > 60 (CDC-valid) | Retained and documented |
| Binary columns | All confirmed 0/1 only | No action required |
| Data types | All float64 (appropriate) | No conversion needed |

The dataset is clean and ready for EDA.

---
## 2. Exploratory Data Analysis (EDA)

### 2.1 Class distribution (target variable)

In [ ]:
sns.countplot(x='Diabetes_binary', hue = 'Diabetes_binary', palette=['#5A95FA', '#FF859B'], data=df)
plt.title("Class Distribution")
plt.show()

print(df['Diabetes_binary'].value_counts(normalize=True))

> **Note:**
> Dataset is balanced. No further action is needed

### 2.2 Distribution of target features (BMI, Age, HighBP, HighChol)


In [ ]:
sns.histplot(df['BMI'], bins=30, kde=True)
plt.title('BMI distribution')
plt.show()

In [ ]:
sns.countplot(data = df, x = 'Age')
plt.title('Age distribution')
plt.show()

In [ ]:
binary_features = [
    "HighBP", "HighChol"
]

for col in binary_features:
    sns.countplot(data=df, x = col, hue = col)
    plt.title(f'{col} distribution')
    plt.show()

### 2.3 Relationship between target variables vs Diabetes outcome


In [ ]:
# Explore relationship between BMI and diabetes status
sns.violinplot(x='Diabetes_binary', hue = 'Diabetes_binary',y='BMI', data=df)
plt.show()

Note: Median of diabetic group is higher than non-diabetic group

For each binary variable, we compare the **diabetic rate** between those with (1) and
without (0) the condition.

In [ ]:
rates = {}

for col in binary_features:
    rate_1 = df[df[col] == 1]["Diabetes_binary"].mean() * 100
    rate_0 = df[df[col] == 0]["Diabetes_binary"].mean() * 100

    # Save dict 
    rates[col] = {"Present (=1)": rate_1, "Absent (=0)": rate_0}

    # Plot bar chart
    plt.figure(figsize=(4, 4))
    plt.bar(["Absent (=0)", "Present (=1)"], [rate_0, rate_1],
            color=["#5DCAA5", "#DB5A30"], alpha=0.85)

    plt.title(f"Diabetic rate for {col}")
    plt.ylabel("Diabetic rate (%)")
    plt.ylim(0, 100)
    plt.grid(axis="y", linestyle="--", alpha=0.3)

    plt.show()

# Create table from dict
rates_df = pd.DataFrame(rates).T
rates_df = rates_df.sort_values("Present (=1)", ascending=False)
print(rates_df.round(2))




**Key observations from binary features:**
- `HighBP` and `HighChol` show strong associations with diabetic rate
- `HighBP`: 66.8% diabetic when present vs 28.3% when absent
- `HighChol` has similar pattern with `HighBP`

### 2.4 Subgroup analysis

Examining diabetic rates across age groupings.

In [ ]:
# Age labels
age_labels = {
    1:"18-24", 2:"25-29", 3:"30-34", 4:"35-39", 5:"40-44",
    6:"45-49", 7:"50-54", 8:"55-59", 9:"60-64", 10:"65-69",
    11:"70-74", 12:"75-79", 13:"80+"
}

age_rate = df.groupby("Age")["Diabetes_binary"].mean() * 100
age_rate.index = [age_labels[i] for i in age_rate.index]

plt.figure(figsize=(8,4))
plt.plot(age_rate.index, age_rate.values, marker="o", color="#7F77DD")
plt.title("Diabetic rate by age group")
plt.ylabel("Diabetic rate (%)")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.show()


**Observations:**
Diabetic rate rises steeply with age — from ~10% in the 18–24 group to >60% in the 75–79 group.

### 2.5 Correlation heatmap

In [ ]:
# Heatmap of all variables
plt.figure(figsize=(14, 10))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, annot_kws={"size": 7})
plt.title("Pearson correlation matrix", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of 4 target variables
rq_cols = ["BMI", "Age", "HighBP", "HighChol", "Diabetes_binary"]
corr_rq = df[rq_cols].corr()
sns.heatmap(corr_rq, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, annot_kws={"size": 7})
plt.title("Pearson correlation matrix", fontsize=13)
plt.tight_layout()
plt.show()

**Note for correlations:**
- `HighBP` (0.38) show the strongest positive correlations with diabetes.
- `HighChol`, `BMI` (0.29) and `Age` (0.28) show similar, moderate correlations.
- Low inter-feature correlations confirm that multicollinearity is not a concern for these 4 predictors.

### 3. Building baseline models

In [ ]:
features = ["BMI", "Age", "HighBP", "HighChol"]
scale_col = ["BMI", "Age"]          # continuous/ordinal
not_scale_col = ["HighBP", "HighChol"]  # binary (0/1)
outcome = "Diabetes_binary"

X = df[features].copy()
y = df['Diabetes_binary'].copy()

In [ ]:
# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42, stratify=y)

In [ ]:
#Scaling process
preprocessor = ColumnTransformer([
    ("scale", StandardScaler(), scale_col),       
    ("not_scale", "passthrough", not_scale_col), 
])

preprocessor.fit(X_train)
X_train_processed = preprocessor.transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

In [ ]:
baseline_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": CalibratedClassifierCV(LinearSVC(random_state=42), cv=3)
}

for name, model in baseline_models.items():
    model.fit(X_train_processed, y_train)

    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:, 1]

    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("Specificity:", recall_score(y_test, y_pred, pos_label=0))
    print("F1:", f1_score(y_test, y_pred))
    print("AUC:", roc_auc_score(y_test, y_prob))


### Next step: Hyperparameter tuning